# AI Movie Studio — LOCAL AI / NO API KEYS
All inference runs inside this Colab runtime. Model weights download only when their engine is first used and are cached on Google Drive. No hosted inference APIs or API keys.

**Use a GPU runtime.** Models are loaded one at a time and released after each job. First use of an engine downloads its weights; later runs reuse Drive cache. Colab Free GPU/session availability is not guaranteed.


In [ ]:
from pathlib import Path
import os, subprocess, sys
repo = Path('/content/ai-movie-studio')
if not repo.exists():
    subprocess.run(['git','clone','https://github.com/PandaCwel/Ai-video-maker.git',str(repo)],check=True)
os.chdir(repo)
print('Repository:', repo)


In [ ]:
!apt-get -qq update && apt-get -qq install -y ffmpeg
!pip -q install -r requirements.txt
!pip -q install -r requirements-local.txt


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
data_root = Path('/content/drive/MyDrive/AI-Movie-Studio')
(data_root/'data').mkdir(parents=True,exist_ok=True)
(data_root/'models').mkdir(parents=True,exist_ok=True)
(data_root/'models'/'huggingface').mkdir(parents=True,exist_ok=True)
(data_root/'models'/'torch').mkdir(parents=True,exist_ok=True)
(data_root/'models'/'coqui-tts').mkdir(parents=True,exist_ok=True)
os.environ['STUDIO_DATA_DIR'] = str(data_root/'data')
os.environ['HF_HOME'] = str(data_root/'models'/'huggingface')
os.environ['TORCH_HOME'] = str(data_root/'models'/'torch')
os.environ['TTS_HOME'] = str(data_root/'models'/'coqui-tts')
os.environ.setdefault('LOCAL_LLM_MODEL','Qwen/Qwen2.5-3B-Instruct')
os.environ.setdefault('LOCAL_IMAGE_MODEL','stable-diffusion-v1-5/stable-diffusion-v1-5')
os.environ.setdefault('LOCAL_VIDEO_MODEL','guoyww/animatediff-motion-adapter-v1-5-2')
os.environ.setdefault('LOCAL_MUSIC_MODEL','facebook/musicgen-small')
print('Projects directory:', data_root/'data'/'projects')
print('Hugging Face model cache:', os.environ['HF_HOME'])
print('Coqui XTTS cache:', os.environ['TTS_HOME'])
print('Model weights download only on first use of each engine; empty cache before first generation is expected.')
print('Projects and downloaded model cache will persist in Google Drive. No API keys required.')


In [ ]:
import torch
print('PyTorch:',torch.__version__)
print('CUDA:',torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:',torch.cuda.get_device_name(0))
    print('VRAM GB:',round(torch.cuda.get_device_properties(0).total_memory/1024**3,1))
else:
    print('WARNING: CPU inference is possible for some models but video generation will be extremely slow.')


## Required local lip-sync (Wav2Lip)
Full production is blocked unless Wav2Lip is installed and its checkpoint is present. The notebook clones the official code and downloads the published GAN checkpoint plus face-detection weights into Google Drive. **Important:** the official open-source Wav2Lip implementation and pretrained model are restricted to personal/research/non-commercial use; do not use them commercially without permission. See the [official license and instructions](https://github.com/Rudrabha/Wav2Lip#license-and-citation).


In [ ]:
# Install Wav2Lip once; keep source and weights on Google Drive.
import urllib.request
wav2lip_repo = repo/'vendor'/'Wav2Lip'
if not (wav2lip_repo/'inference.py').exists():
    wav2lip_repo.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['git','clone','--depth','1','https://github.com/Rudrabha/Wav2Lip.git',str(wav2lip_repo)],check=True)
!pip -q install 'librosa==0.10.2.post1' gdown opencv-contrib-python-headless
wav2lip_checkpoint = data_root/'models'/'wav2lip_gan.pth'
if not wav2lip_checkpoint.exists():
    subprocess.run([sys.executable,'-m','gdown','--id','15G3U08c8xsCkOqQxE38Z2XXDnPcOptNk','-O',str(wav2lip_checkpoint)],check=True)
s3fd_checkpoint = wav2lip_repo/'face_detection'/'detection'/'sfd'/'s3fd.pth'
s3fd_checkpoint.parent.mkdir(parents=True,exist_ok=True)
if not s3fd_checkpoint.exists():
    urllib.request.urlretrieve('https://www.adrianbulat.com/downloads/python-fan/s3fd-619a316812.pth',s3fd_checkpoint)
os.environ['WAV2LIP_CHECKPOINT'] = str(wav2lip_checkpoint)
os.environ['WAV2LIP_REPO'] = str(wav2lip_repo)
assert wav2lip_checkpoint.is_file() and s3fd_checkpoint.is_file() and (wav2lip_repo/'inference.py').is_file()
print('Wav2Lip ready:', wav2lip_checkpoint, '| face detector:', s3fd_checkpoint)


In [ ]:
import subprocess, time, urllib.request, urllib.error, os, sys

# Stop an older server instance from a previous run, if one exists.
try:
    server.terminate()
    server.wait(timeout=5)
except Exception:
    pass

server_log = open('/content/ai-movie-studio-uvicorn.log', 'w')
server = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'app.main:app', '--host', '0.0.0.0', '--port', '8000', '--log-level', 'info'],
    cwd=str(repo), stdout=server_log, stderr=subprocess.STDOUT
)

# Do not publish a proxy URL until the app really responds on localhost.
ready = False
last_error = None
for attempt in range(90):
    if server.poll() is not None:
        break
    try:
        with urllib.request.urlopen('http://127.0.0.1:8000/api/health', timeout=2) as response:
            health_payload = response.read().decode('utf-8')
            if response.status == 200:
                ready = True
                break
    except Exception as exc:
        last_error = repr(exc)
    time.sleep(2)

server_log.flush()
if not ready:
    server_log.close()
    print('❌ AI Movie Studio nie uruchomiło się poprawnie.')
    print('Ostatni błąd połączenia:', last_error)
    print('Log serwera:')
    print(Path('/content/ai-movie-studio-uvicorn.log').read_text(errors='replace')[-12000:])
    raise RuntimeError('Uvicorn nie odpowiada na http://127.0.0.1:8000/api/health — sprawdź log powyżej.')

with urllib.request.urlopen('http://127.0.0.1:8000/', timeout=10) as response:
    page = response.read()
    print('Strona główna:', response.status, response.headers.get('Content-Type'), 'rozmiar:', len(page), 'B')
    if not page or b'AI MOVIE' not in page.upper():
        raise RuntimeError('Serwer działa, ale strona główna jest pusta lub nieprawidłowa.')

for asset_path in ['/static/css/studio.css', '/static/js/studio.js']:
    with urllib.request.urlopen('http://127.0.0.1:8000' + asset_path, timeout=10) as asset_response:
        asset_body = asset_response.read()
        if asset_response.status != 200 or not asset_body:
            raise RuntimeError(f'Brak zasobu frontendu: {asset_path}')
        print('Frontend asset OK:', asset_path, len(asset_body), 'B')

from google.colab.output import eval_js
studio_url = eval_js('google.colab.kernel.proxyPort(8000)')
print('✅ AI Movie Studio:', studio_url)
print('API health:', studio_url + '/api/health')
print('API docs:', studio_url + '/docs')
print('Jeśli strona nie otwiera się w aplikacji Colab, skopiuj link i otwórz go w Safari/Chrome.')


## Local engines and model lifecycle
- **Screenplay / AI Director:** Qwen 2.5 3B Instruct; long scripts are analyzed in ordered chunks, without a fixed 50k truncation.
- **Images:** Stable Diffusion 1.5, downloaded on first image request.
- **Video:** AnimateDiff + SD 1.5, downloaded on first video request.
- **Voice:** XTTS-v2, downloaded on first speech request.
- **Lip-sync:** required local Wav2Lip; code, GAN checkpoint and S3FD face detector are installed by the notebook.
- **Music:** MusicGen-small, downloaded on first soundtrack request.
- **Editing:** FFmpeg joins clips and mixes generated score beneath dialogue.

Only the engine needed for the current task is loaded. Heavy model references are released and CUDA cache is cleared after each job. Weights stay cached on Drive. Full production requires voice profiles whose names match screenplay character names, dialogue, FFmpeg and a working Wav2Lip installation.

Colab Free has limited and non-guaranteed GPU time/VRAM. A 25–30 minute film is a long sequential render and may require multiple sessions; project data and checkpoints are stored on Drive to support manually resuming. Wav2Lip's open-source model is for personal/research/non-commercial use only unless separate permission is obtained.